# Dugong Watch — Track C: Habitat-Risk Layer & GIS
### `C_risk_gis/Dugong_Habitat_Risk.ipynb`

Turns Track B's seagrass outputs + real regional threat data into a single, ranked
**"risk to dugong habitat"** map for the **Marawah / Bu Tinah** region (Abu Dhabi).

**Runs end-to-end in Google Earth Engine (Colab)** — same stack as `M2_classification/`.

| Task | Section |
|------|---------|
| **C1** Collect threat data | §3 |
| **C2** Risk factors + transparent weighting | §4 (+ `RISK_METHODOLOGY.md`) |
| **C3** Build the risk index | §5 |
| **C4** Ranked "risk to dugong habitat" map | §6 |
| **C5** Sanity-check hotspots vs. real threats | §7 |
| **C6** Hand risk layers to Track D (dashboard) | §8 |

**Model (see `RISK_METHODOLOGY.md`):**  `Risk = Value × Threat`, where
`Value = seagrass × dugong-use` and `Threat = Σ wᵢ·Threatᵢ` (weighted overlay).
Both must be present for a pixel to score high — this is *exposure × consequence*
habitat-risk logic (InVEST HRA; Halpern et al. 2008).

## 0 — Setup
Run first, every session (mirrors `A2_SetupEnvironment.ipynb`).

In [ ]:
# Cell 1: Install libraries
!pip install earthengine-api geemap geopandas --quiet

In [ ]:
# Cell 2: Imports
import ee, geemap, json, os
import pandas as pd

In [ ]:
# Cell 3: Authenticate Earth Engine
ee.Authenticate()

In [ ]:
# Cell 4: CONFIG  ───────────────────────────────────────────────────────────
# Set your own GEE project ID (same one you use for Track B).
PROJECT = "your-gee-project-id"          # <-- EDIT ME

# --- Track B handoff --------------------------------------------------------
# Path A (preferred): you ingested Track B's exported GeoTIFFs as EE assets.
#   Set USE_TRACKB_ASSETS = True and paste the asset IDs below.
# Path B (default): regenerate the 2024 habitat map + 2019->2024 change map
#   inline from the repo's EAD training polygons, using Track B's exact
#   pipeline (deterministic, seed=42). Keeps Track C fully stand-alone.
USE_TRACKB_ASSETS = False
HABITAT_ASSET = "projects/your-gee-project-id/assets/rf_final_2024"
CHANGE_ASSET  = "projects/your-gee-project-id/assets/seagrass_change_2019_2024"

# --- Time windows (match Track B) ------------------------------------------
CURRENT_START, CURRENT_END = "2024-01-01", "2024-12-31"
PAST_START,    PAST_END    = "2019-01-01", "2019-12-31"

# --- C2 risk weights (must sum to 1.0) — see RISK_METHODOLOGY.md ------------
WEIGHTS = {
    "loss":        0.30,   # T1 observed seagrass loss (Track B change map)
    "development": 0.20,   # T2 coastal development / dredging
    "thermal":     0.20,   # T3 SST thermal stress (Landsat)
    "discharge":   0.15,   # T4 desalination / industrial discharge
    "vessel":      0.15,   # T5 vessel / boat pressure
}
assert abs(sum(WEIGHTS.values()) - 1.0) < 1e-9, "Weights must sum to 1.0"

# --- Distance-decay lengths λ (metres): pressure = exp(-d/λ) ----------------
DECAY_M = {"development": 3000, "discharge": 5000, "vessel": 4000, "dugong": 12000}

# --- Repo paths (works whether run from repo root or C_risk_gis/) -----------
def find_path(cands):
    p = next((c for c in cands if os.path.exists(c)), None)
    if p is None:
        raise FileNotFoundError(f"none found: {cands}")
    return p

STUDY_AREA_PATH = find_path([
    "/content/study_area.geojson",
    "../data/study_area/study_area.geojson",
    "data/study_area/study_area.geojson",
])
THREAT_DIR = find_path(["threat_data", "C_risk_gis/threat_data", "../C_risk_gis/threat_data"])
TRAIN_DIR  = find_path(["../data/training", "data/training", "/content"])
print("study area :", STUDY_AREA_PATH)
print("threat dir :", THREAT_DIR)
print("train dir  :", TRAIN_DIR)

In [ ]:
# Cell 5: Initialize Earth Engine
ee.Initialize(project=PROJECT)
print("Earth Engine initialized on project:", PROJECT)

In [ ]:
# Cell 6: Load study area (AOI)
with open(STUDY_AREA_PATH) as f:
    geojson = json.load(f)
study_area = ee.Geometry(geojson["features"][0]["geometry"])
print("Study area loaded. Area (km2):",
      round(study_area.area().divide(1e6).getInfo(), 1))

## 1 — Track B handoff: seagrass habitat + change

We need two rasters from Track B on the 10 m grid:
* **seagrass presence** (2024) — `1` = seagrass, `0` = non-seagrass → the *Value* base.
* **seagrass loss** (2019→2024) — `loss` pixels → threat **T1**.

If `USE_TRACKB_ASSETS` is `True` we load Track B's exact exports. Otherwise we
regenerate them here with Track B's pipeline (same bands, same RF, `seed=42`).

In [ ]:
# Cell 7: Track B pipeline helpers (identical processing to M2_classification)
def mask_s2_clouds(image):
    qa = image.select("QA60")
    mask = (qa.bitwiseAnd(1 << 10).eq(0)
              .And(qa.bitwiseAnd(1 << 11).eq(0)))
    return image.updateMask(mask).divide(10000).copyProperties(image, ["system:time_start"])

PREDICTOR_BANDS = ["B2","B3","B4","B5","B6","B7","B8","B8A","B11","B12",
                   "NDVI","NDWI","MNDWI","BSI"]

def build_feature_stack(start, end, bounds=study_area):
    coll = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
              .filterBounds(bounds).filterDate(start, end)
              .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 20))
              .map(mask_s2_clouds))
    comp = coll.median().clip(bounds)
    ndvi  = comp.normalizedDifference(["B8","B4"]).rename("NDVI")
    ndwi  = comp.normalizedDifference(["B3","B8"]).rename("NDWI")
    mndwi = comp.normalizedDifference(["B3","B11"]).rename("MNDWI")
    bsi = comp.expression(
        "((SWIR + RED) - (NIR + BLUE)) / ((SWIR + RED) + (NIR + BLUE))",
        {"SWIR":comp.select("B11"),"RED":comp.select("B4"),
         "NIR":comp.select("B8"),"BLUE":comp.select("B2")}).rename("BSI")
    return comp.addBands([ndvi,ndwi,mndwi,bsi]).select(PREDICTOR_BANDS)

print("Helpers ready.")

In [ ]:
# Cell 8: Build 2019 + 2024 feature stacks
predictor_t1 = build_feature_stack(PAST_START, PAST_END)      # 2019
predictor_t2 = build_feature_stack(CURRENT_START, CURRENT_END) # 2024
print("Feature stacks built for 2019 and 2024.")

In [ ]:
# Cell 9: Get habitat (2024) + change map — from assets OR regenerated
if USE_TRACKB_ASSETS:
    habitat_2024 = ee.Image(HABITAT_ASSET).rename("classification").clip(study_area)
    change_map   = ee.Image(CHANGE_ASSET).rename("change").clip(study_area)
    print("Loaded Track B assets.")
else:
    # Regenerate with Track B's RF, trained on the repo's EAD training polygons.
    seagrass_fc = geemap.vector_to_ee(
        find_path([f"{TRAIN_DIR}/marawah_seagrass_training.geojson",
                   "/content/marawah_seagrass_training.geojson"]))
    nonseag_fc  = geemap.vector_to_ee(
        find_path([f"{TRAIN_DIR}/marawah_non_seagrass_training.geojson",
                   "/content/marawah_non_seagrass_training.geojson"]))

    # Stratified random points inside the training polygons (don't treat 1 polygon = 1 sample)
    pts_1 = ee.FeatureCollection.randomPoints(seagrass_fc.geometry(), 800, 42) \
              .map(lambda f: f.set("class", 1))
    pts_0 = ee.FeatureCollection.randomPoints(nonseag_fc.geometry(), 800, 42) \
              .map(lambda f: f.set("class", 0))
    training_pts = pts_1.merge(pts_0)

    training = predictor_t2.sampleRegions(
        collection=training_pts, properties=["class"], scale=10, tileScale=4)
    rf = ee.Classifier.smileRandomForest(numberOfTrees=100, seed=42) \
           .train(features=training, classProperty="class", inputProperties=PREDICTOR_BANDS)

    habitat_2024 = predictor_t2.classify(rf).rename("classification").clip(study_area)
    habitat_2019 = predictor_t1.classify(rf).rename("classification").clip(study_area)

    # change codes: 0 stable non-seagrass, 1 stable seagrass, 2 gain, 3 loss
    code_img   = habitat_2019.multiply(10).add(habitat_2024)
    change_map = code_img.remap([0,11,1,10],[0,1,2,3]).rename("change").clip(study_area)
    print("Regenerated habitat_2024 + change_map with Track B's pipeline (seed=42).")

# The two layers Track C consumes:
seagrass_presence = habitat_2024.eq(1).rename("seagrass")   # 1 where seagrass
seagrass_loss     = change_map.eq(3).rename("loss")         # 1 where lost 2019->2024

In [ ]:
# Cell 10: Quick look at the Track B handoff
Map = geemap.Map()
Map.add_basemap("SATELLITE")
Map.centerObject(study_area, 11)
Map.addLayer(seagrass_presence.selfMask(), {"palette":["#1b9e77"]}, "Seagrass 2024")
Map.addLayer(change_map, {"min":0,"max":3,
    "palette":["#8c6d4f","#1b9e77","#2c7fb8","#e34a33"]},
    "Change 2019-2024 (loss=red)")
Map.addLayer(ee.Image().byte().paint(study_area,1,2), {"palette":["black"]}, "AOI")
Map.addLayerControl(); Map

## 2 — C1: Threat data

Loaded from `threat_data/` (see `threat_data/SOURCES.md` for every citation). Built
infrastructure and islands are real coordinates; reserve boundary, dugong-density
surface and nav routes are approximate/schematic and flagged in each file.

In [ ]:
# Cell 11: Load threat layers as Earth Engine FeatureCollections
def load_fc(name):
    return geemap.vector_to_ee(f"{THREAT_DIR}/{name}")

protected   = load_fc("protected_areas.geojson")
dugong      = load_fc("dugong_areas.geojson")
discharge   = load_fc("desalination_industrial.geojson")
development = load_fc("coastal_development.geojson")
vessel      = load_fc("vessel_pressure.geojson")

for nm, fc in [("protected",protected),("dugong",dugong),("discharge",discharge),
               ("development",development),("vessel",vessel)]:
    print(f"{nm:12s}: {fc.size().getInfo()} features")

In [ ]:
# Cell 12: Map the threat context
Map = geemap.Map(); Map.add_basemap("SATELLITE"); Map.centerObject(study_area, 10)
Map.addLayer(protected, {"color":"00FF00"}, "Protected areas")
Map.addLayer(dugong.style(**{"color":"1f78b4","pointSize":8}), {}, "Dugong areas")
Map.addLayer(discharge.style(**{"color":"e34a33","pointSize":8}), {}, "Desal/industrial")
Map.addLayer(development.style(**{"color":"ff7f00","pointSize":8}), {}, "Coastal dev/dredging")
Map.addLayer(vessel.style(**{"color":"6a3d9a","pointSize":6}), {}, "Vessel pressure")
Map.addLayer(ee.Image().byte().paint(study_area,1,2), {"palette":["yellow"]}, "AOI")
Map.addLayerControl(); Map

## 3 — C2: Build the normalized Value & Threat factors

Every factor is rescaled to **0–1 across the AOI** before it is combined, so no factor
wins just because of its units. Helpers below do min–max normalization (server-side)
and `exp(-d/λ)` distance decay.

In [ ]:
# Cell 13: Helper — normalize an image to 0-1 over the AOI (server-side)
def normalize(img, scale=30):
    mm = img.reduceRegion(reducer=ee.Reducer.minMax(), geometry=study_area,
                          scale=scale, maxPixels=1e13, bestEffort=True)
    band = img.bandNames().get(0)
    lo = ee.Number(mm.get(ee.String(band).cat("_min")))
    hi = ee.Number(mm.get(ee.String(band).cat("_max")))
    return img.subtract(lo).divide(hi.subtract(lo).max(1e-6)).clamp(0,1).clip(study_area)

# Helper — distance decay from a FeatureCollection: exp(-distance/lambda), 0..1
def decay(fc, lam_m, search_m=100000):
    d = fc.distance(searchRadius=search_m).clip(study_area)     # metres to nearest feature
    return d.divide(lam_m).multiply(-1).exp().unmask(0).clamp(0,1).rename("decay")

print("Helpers ready.")

In [ ]:
# Cell 14: VALUE layer  = seagrass presence x dugong-use weight
dugong_use = normalize(decay(dugong, DECAY_M["dugong"]))          # 0..1, peaks at cores
value = seagrass_presence.multiply(dugong_use.multiply(0.4).add(0.6)).rename("value")
# non-seagrass -> ~0 ; seagrass in dugong core -> up to ~1.0 ; reserve-edge seagrass -> 0.6
print("Value layer built (seagrass x [0.6 + 0.4*dugong_use]).")

In [ ]:
# Cell 15: THREAT factors T1-T5, each normalized 0-1
# T1 — observed seagrass loss, smoothed into a continuous pressure surface
loss_density = seagrass_loss.reduceNeighborhood(
    reducer=ee.Reducer.mean(), kernel=ee.Kernel.circle(10, "pixels")).rename("loss")
T1_loss = normalize(loss_density)

# T2 — coastal development / dredging (distance decay)
T2_dev = decay(development, DECAY_M["development"]).rename("dev")

# T4 — desalination / industrial discharge (distance decay)
T4_dis = decay(discharge, DECAY_M["discharge"]).rename("dis")

# T5 — vessel / boat pressure (distance decay)
T5_ves = decay(vessel, DECAY_M["vessel"]).rename("ves")
print("T1, T2, T4, T5 built. (T3 thermal next.)")

In [ ]:
# Cell 16: T3 — thermal stress from Landsat 8/9 summer surface temperature
def landsat_st(coll_id):
    return (ee.ImageCollection(coll_id)
              .filterBounds(study_area)
              .filterDate("2020-01-01","2024-12-31")
              .filter(ee.Filter.calendarRange(6, 9, "month"))   # Jun-Sep (hottest)
              .filter(ee.Filter.lt("CLOUD_COVER", 20)))

l89 = landsat_st("LANDSAT/LC08/C02/T1_L2").merge(landsat_st("LANDSAT/LC09/C02/T1_L2"))

def to_celsius(img):
    return img.select("ST_B10").multiply(0.00341802).add(149.0).subtract(273.15) \
              .rename("sst").copyProperties(img, ["system:time_start"])

# Mask SST to water only (MNDWI > 0 on the 2024 S2 composite) BEFORE normalize(),
# so hot dry land can't stretch the min-max scaling of the thermal-stress layer.
water_2024 = predictor_t2.select("MNDWI").gt(0)
sst = l89.map(to_celsius).median().updateMask(water_2024).clip(study_area)
T3_thermal = normalize(sst).rename("thermal")
print("T3 thermal built from Landsat ST_B10 (Jun-Sep median, water-masked, hotter = higher stress).")

In [ ]:
# Cell 17: C2 — weighted overlay -> total threat pressure (0-1)
threat = (T1_loss.multiply(WEIGHTS["loss"])
          .add(T2_dev.multiply(WEIGHTS["development"]))
          .add(T3_thermal.multiply(WEIGHTS["thermal"]))
          .add(T4_dis.multiply(WEIGHTS["discharge"]))
          .add(T5_ves.multiply(WEIGHTS["vessel"]))
          ).rename("threat").clip(study_area)
print("Threat = " + " + ".join(f"{w:.2f}*{k}" for k,w in WEIGHTS.items()))

## 4 — C3: The Dugong-Habitat Risk Index

`RiskRaw = Value × Threat`, rescaled to 0–1. Multiplying enforces the rule that risk is
high **only** where valuable dugong habitat and real pressure overlap.

In [ ]:
# Cell 18: C3 — risk index
risk_raw = value.multiply(threat).rename("risk_raw")
risk = normalize(risk_raw).rename("risk")           # 0..1 relative risk within AOI
print("Risk index built.")

In [ ]:
# Cell 19: 5 ranked risk classes (quantile breaks, seagrass area only)
pct = risk.updateMask(seagrass_presence).reduceRegion(
    reducer=ee.Reducer.percentile([20,40,60,80]),
    geometry=study_area, scale=30, maxPixels=1e13, bestEffort=True).getInfo()
b = [pct.get("risk_p20"), pct.get("risk_p40"), pct.get("risk_p60"), pct.get("risk_p80")]
# Guard: a percentile can come back None if the masked region is empty.
print("Class breaks (risk):", [round(x, 3) if x is not None else None for x in b])
assert all(x is not None for x in b), "Percentile breaks are null — check seagrass mask / AOI."

risk_class = (ee.Image(1)
    .where(risk.gt(b[0]), 2).where(risk.gt(b[1]), 3)
    .where(risk.gt(b[2]), 4).where(risk.gt(b[3]), 5)
    .updateMask(seagrass_presence).rename("risk_class").clip(study_area))
# 1 Very Low .. 5 Very High (only over seagrass, since that is what is at stake)

## 5 — C4: "Risk to dugong habitat" map + ranked hotspots

In [ ]:
# Cell 20: Risk map
RISK_PALETTE = ["#2c7bb6","#abd9e9","#ffffbf","#fdae61","#d7191c"]  # blue->red
Map = geemap.Map(); Map.add_basemap("SATELLITE"); Map.centerObject(study_area, 11)
Map.addLayer(risk_class, {"min":1,"max":5,"palette":RISK_PALETTE}, "Risk to dugong habitat")
Map.addLayer(discharge.style(**{"color":"black","pointSize":7}), {}, "Discharge pts")
Map.addLayer(development.style(**{"color":"black","pointSize":7}), {}, "Development pts")
Map.addLayer(ee.Image().byte().paint(study_area,1,2), {"palette":["white"]}, "AOI")
try:
    Map.add_legend(title="Dugong-habitat risk",
        labels=["Very low","Low","Moderate","High","Very high"],
        colors=RISK_PALETTE)
except Exception as e:
    print("legend:", e)
Map.addLayerControl(); Map

In [ ]:
# Cell 21: C4 — extract & rank hotspots (top risk class, dissolved to polygons)
hotspots = risk_class.eq(5).selfMask().reduceToVectors(
    geometry=study_area, scale=30, geometryType="polygon",
    eightConnected=True, maxPixels=1e13, labelProperty="zone")

# keep meaningful clusters (>= 5 ha), attach area + mean risk + dominant threat
MIN_HA = 5
def enrich(f):
    geom = f.geometry()
    area_ha = geom.area(maxError=1).divide(1e4)
    mr = risk.reduceRegion(ee.Reducer.mean(), geom, 30, maxPixels=1e13, bestEffort=True).get("risk")
    # dominant threat factor at the hotspot
    tvals = ee.Image.cat([
        T1_loss.multiply(WEIGHTS["loss"]).rename("loss"),
        T2_dev.multiply(WEIGHTS["development"]).rename("development"),
        T3_thermal.multiply(WEIGHTS["thermal"]).rename("thermal"),
        T4_dis.multiply(WEIGHTS["discharge"]).rename("discharge"),
        T5_ves.multiply(WEIGHTS["vessel"]).rename("vessel"),
    ]).reduceRegion(ee.Reducer.mean(), geom, 30, maxPixels=1e13, bestEffort=True)
    return f.set({"area_ha": area_ha, "mean_risk": mr,
                  "loss": tvals.get("loss"), "development": tvals.get("development"),
                  "thermal": tvals.get("thermal"), "discharge": tvals.get("discharge"),
                  "vessel": tvals.get("vessel")})

hotspots = hotspots.map(enrich).filter(ee.Filter.gte("area_ha", MIN_HA))
# priority score = mean risk weighted by patch size
hotspots = hotspots.map(lambda f: f.set(
    "priority", ee.Number(f.get("mean_risk")).multiply(ee.Number(f.get("area_ha")).sqrt())))
hotspots = hotspots.sort("priority", False)
print("Hotspot patches (>= {} ha):".format(MIN_HA), hotspots.size().getInfo())

In [ ]:
# Cell 22: Ranked hotspot table (top 15)
def hs_row(i, f):
    f = ee.Feature(f)
    tw = {k: f.get(k) for k in ["loss","development","thermal","discharge","vessel"]}
    return {"rank": i+1,
            "area_ha": round(ee.Number(f.get("area_ha")).getInfo(), 1),
            "mean_risk": round(ee.Number(f.get("mean_risk")).getInfo(), 3),
            "dominant_threat": max(tw, key=lambda k: ee.Number(f.get(k)).getInfo()),
            "lon": round(f.geometry().centroid(1).coordinates().get(0).getInfo(), 4),
            "lat": round(f.geometry().centroid(1).coordinates().get(1).getInfo(), 4)}

hs_list = hotspots.limit(15).toList(15)
rows = [hs_row(i, hs_list.get(i)) for i in range(min(15, hotspots.size().getInfo()))]
hotspot_df = pd.DataFrame(rows)
hotspot_df

## 6 — C5: Sanity checks & weight sensitivity

Three checks: (1) do hotspots sit near real threats? (2) is core habitat flagged where
pressure reaches it? (3) which hotspots survive re-weighting?

In [ ]:
# Cell 23: (1) Face validity — mean risk vs. distance to nearest discharge point
# Bin distance into 2-km integer classes; never group a reducer on a continuous band.
d_km    = discharge.distance(50000).clip(study_area).divide(1000)          # km to discharge
km_band = d_km.divide(2).floor().multiply(2).toInt().rename("km_band")     # 0,2,4,... km bins
band_stats = risk.updateMask(seagrass_presence).addBands(km_band).reduceRegion(
    reducer=ee.Reducer.mean().group(groupField=1, groupName="km_band"),
    geometry=study_area, scale=60, maxPixels=1e13, bestEffort=True)
print("If the model is sane, mean risk should fall as distance-to-discharge grows.")
print(band_stats.getInfo())

In [ ]:
# Cell 24: (2) Zonal mean risk inside the dugong core vs. the whole AOI
# Build the "core" zone from the very-high-density dugong points (buffered 5 km,
# clipped to the AOI). We do NOT use the Bu Tinah protected polygon: it sits
# ~35 km north of the AOI, so masking risk to it gives an empty region -> null crash.
core_pts = dugong.filter(ee.Filter.eq("density", "very_high"))
core = core_pts.geometry().buffer(5000).intersection(study_area, maxError=1)

def zonal(mask_geom, label):
    # .get("risk") is a computed object; .getInfo() resolves to None cleanly if the
    # region is empty -> no ee.Number(null), no crash.
    v = (risk.updateMask(seagrass_presence)
             .reduceRegion(ee.Reducer.mean(), mask_geom, 30,
                           maxPixels=1e13, bestEffort=True)
             .get("risk")).getInfo()
    shown = round(v, 3) if v is not None else "n/a (empty region)"
    print(f"  mean seagrass risk {label:16s}: {shown}")

print("Zonal risk check:")
zonal(core, "in dugong core")
zonal(study_area, "whole AOI")

In [ ]:
# Cell 25: (3) Weight sensitivity — perturb each weight +/-0.10, re-rank hotspots
def risk_for_weights(w):
    t = (T1_loss.multiply(w["loss"]).add(T2_dev.multiply(w["development"]))
         .add(T3_thermal.multiply(w["thermal"])).add(T4_dis.multiply(w["discharge"]))
         .add(T5_ves.multiply(w["vessel"])))
    return normalize(value.multiply(t))

def top_class_area(risk_img):
    # Rename to "risk" so reducer keys are predictable even for the perturbed layers
    # (value.multiply(t) is band-named "value" -> percentile would be "value_p80").
    risk_img = risk_img.rename("risk")
    p80 = risk_img.updateMask(seagrass_presence).reduceRegion(
        ee.Reducer.percentile([80]), study_area, 60,
        maxPixels=1e13, bestEffort=True).get("risk_p80")
    top = risk_img.gt(ee.Number(p80)).updateMask(seagrass_presence)
    a = top.multiply(ee.Image.pixelArea()).reduceRegion(
        ee.Reducer.sum(), study_area, 60,
        maxPixels=1e13, bestEffort=True).get("risk")
    return round((ee.Number(a).getInfo() or 0)/1e4, 1)  # ha

base_area = top_class_area(risk)
print(f"Baseline high-risk area: {base_area} ha")
for k in WEIGHTS:
    for delta in (+0.10, -0.10):
        w = dict(WEIGHTS); w[k] = max(0, w[k] + delta)
        s = sum(w.values()); w = {kk: vv/s for kk, vv in w.items()}   # renormalize
        a = top_class_area(risk_for_weights(w))
        print(f"  {k:12s} {'+' if delta>0 else '-'}0.10 -> high-risk {a:8} ha "
              f"({(a-base_area)/base_area*100:+.0f}%)")
print("Small % changes => hotspots are robust to the exact weights.")

## 7 — C6: Hand risk layers to Track D (dashboard)

Exports to Google Drive `dugong_uae/` (same folder Track B uses). Schema + symbology
for M4 are in this module's `README.md`. After the tasks finish, share the folder with
Track D (motasem).

In [ ]:
# Cell 26: Export rasters (risk + risk classes) to Drive
def export_img(img, name, scale=10):
    ee.batch.Export.image.toDrive(
        image=img, description=name, folder="dugong_uae",
        fileNamePrefix=name, region=study_area, scale=scale, maxPixels=1e13).start()
    print("export started:", name)

export_img(risk.multiply(1000).toInt16(), "dugong_risk_index")   # 0..1000 (int16, small)
export_img(risk_class.toByte(),           "dugong_risk_class")   # 1..5
export_img(value.multiply(1000).toInt16(),"dugong_value_layer")  # for dashboard drill-down
export_img(threat.multiply(1000).toInt16(),"dugong_threat_layer")

In [ ]:
# Cell 27: Export ranked hotspots + threat layers as GeoJSON (vector) to Drive
ee.batch.Export.table.toDrive(
    collection=hotspots, description="dugong_risk_hotspots",
    folder="dugong_uae", fileNamePrefix="dugong_risk_hotspots",
    fileFormat="GeoJSON").start()
print("export started: dugong_risk_hotspots.geojson")

for nm, fc in [("threat_protected_areas",protected),("threat_dugong_areas",dugong),
               ("threat_discharge",discharge),("threat_development",development),
               ("threat_vessel",vessel)]:
    ee.batch.Export.table.toDrive(collection=fc, description=nm, folder="dugong_uae",
                                  fileNamePrefix=nm, fileFormat="GeoJSON").start()
    print("export started:", nm)

In [ ]:
# Cell 28: Save the ranked hotspot table (CSV) for the write-up / dashboard
hotspot_df.to_csv("dugong_risk_hotspots_ranked.csv", index=False)
print("saved: dugong_risk_hotspots_ranked.csv")

# The CSV above only lives in this Colab session's local disk - it disappears
# when the runtime disconnects. Trigger a real download so it actually leaves
# the session (works the same on any OS, since it's a plain browser download).
try:
    from google.colab import files
    files.download("dugong_risk_hotspots_ranked.csv")
    print("download triggered - check your browser's Downloads.")
except ImportError:
    print("not running in Colab - file is next to the notebook.")

print()
print("Track C outputs -> Drive/dugong_uae:")
print("  dugong_risk_index.tif   (0-1000 int, relative risk)")
print("  dugong_risk_class.tif   (1-5 ranked classes)")
print("  dugong_value_layer.tif  / dugong_threat_layer.tif")
print("  dugong_risk_hotspots.geojson (ranked, with area/mean_risk/dominant_threat)")
print("  threat_*.geojson (context layers)")
print()
print("Check Earth Engine > Tasks to confirm completion, then share with Track D.")

---
### Handoff summary (C6)
Track D loads `dugong_risk_class.tif` as the main dashboard layer (blue→red, 5 classes),
`dugong_risk_hotspots.geojson` as ranked, clickable markers (popup: rank, area, mean risk,
dominant threat), and the `threat_*.geojson` as toggleable context. `dugong_value_layer`
and `dugong_threat_layer` power the "why is it risky here?" drill-down. Full field schema
and colour ramps are in `C_risk_gis/README.md`.